# Experiment 08 — score forecasts and vault admission thresholds

Based on [07-opt-score-position-count.ipynb](07-opt-score-position-count.ipynb), with frozen scoring methods from experiments 04–07. Implementation follows [the reviewed plan](08-forecast-score-threshold-plan.md), reviewed with Claude Opus 5.5.

## User goal

Understand how the different scoring methods relate to future vault profitability and whether there is a score threshold below which vaults should no longer be admitted. Forecast **every four calendar days**. For **every scoring method × 7/30/90-day lookahead**, show score on X and realised **CAGR and Sharpe** on Y, trend lines and points coloured by forecast-origin quarter. Analyse the strength and stability of the relationships and test additional ways to understand the data.

## Method

Inventory every distinct implemented composite recipe, including the 74 frozen two-/three-/four-input supports and older clipping/missingness alternatives. Compute scores from completed T−1 features and use calendar-aligned future per-share NAV endpoints. All metrics use the latest observed positive NAV carried across missing calendar days. Never backfill before first observation or create labels after last observation/archive end. Do not replace unknown scores with zero or treat filled historical marks as actual observations. Analyse same-date rank IC, uncertainty, calibration, clipping, missingness, tails, quarters, delayed anchors, lagged-score placebos and incremental inputs.

Freeze the monthly threshold rule before seeing outcomes. Compare source v11, two-input Blend 46d and the focal four-input method at 6/7 positions, with separate slot-to-cash and source capped-renormalisation treatments. Keep the original two-day trading cadence, inverse variance, selection hysteresis, guards and source-modelled costs. Short-period **2026-01-01 → 2026-10-05 (end exclusive, UTC)** is the default throughout; full history is secondary and overlaps previously researched dates.

## Key new insights

- Recomputed all 139 score recipes at 7/30/90-day horizons. Primary targets use the latest actual positive NAV carried forward on calendar days; observation counts remain separate. No leading backfill or labels beyond each vault’s last actual mark/archive end. The source score and portfolio machinery is unchanged.
- Primary 30-day return rank IC ranges from **0.087 to 0.089**; Sharpe rank IC ranges from **0.129 to 0.137** across the three frozen methods. Inspect the intervals and origin counts below before interpreting these point estimates.
- Of 54 supported linear/isotonic method–horizon–target comparisons, **35** have lower equal-date MSE than the same-row historical constant. This does not by itself establish forecast or portfolio skill.
- The absolute floor is active in **27/30** 2026 method-months and rejects **0** proposed entries across the replay treatments.
- The highest non-random replay is **focal_four, 6 positions, none / slot_to_cash**: CAGR **51.77%**, Sharpe **2.30**, drawdown **-6.35%**. This is a descriptive maximum on previously researched history.

See [updated findings](08-score-forecast-findings.md) and the [sparse NAV audit](sparse-nav-rerun-2026-10-10.md).

## Summary of results

The tables below distinguish forecast ranking, calibration and actual executable portfolio performance. The three focal methods have separate scatter views after the input checks. The complete catalogue contains every method/lookahead combination in small, separately linked notebooks.

## Robustness of results

Checks include independent raw-label recomputation, original source-score equality, full equity/trade baseline matching, actual versus carried NAVs, serial dependence, sparse support, extreme annualisation, per-vault/quarter omissions and PnL concentration. Short-period 90-day estimates are descriptive: confidence intervals and p-values are deliberately absent. Existing methods were selected on already inspected history; this is conditional chronological validation, not an untouched holdout.


## Inputs, observed targets and frozen design

Load the pinned repaired generation, freeze the analysis rules and independently audit raw observations before viewing relationships.

In [1]:
from pathlib import Path
import sys
RESEARCH_DIR = Path.cwd()
if not (RESEARCH_DIR / 'score_forecast_data.py').exists():
    RESEARCH_DIR = RESEARCH_DIR / 'scratchpad' / 'hyperliquid-vov-2nd'
sys.path.insert(0, str(RESEARCH_DIR))
import json
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Markdown
from score_matrix_visualisation import compact_output
from score_forecast_data import prepare_data, ARTIFACT_DIR
from score_forecast_statistics import analyse_data, join_method, freeze_threshold_rule
from score_forecast_execution import freeze_execution_design, run_threshold_experiments
from score_forecast_audit import independent_label_audit
from score_forecast_visualisation import *
from score_forecast_report import *

class Parameters:
    forecast_days = 4
    lookaheads = (7, 30, 90)
    positions = (6, 7)
    threshold_horizon = 30
    random_seeds = (8081, 8082, 8083)
    short_start = pd.Timestamp('2026-01-01')
    long_start = pd.Timestamp('2023-03-01')
    end_exclusive = pd.Timestamp('2026-10-05')

with compact_output('Pinned score/label generation and predeclared policy rules'):
    data = prepare_data()
    threshold_rule = freeze_threshold_rule(data)
    execution_design = freeze_execution_design()
    raw_audit, audit_report = independent_label_audit(data, ARTIFACT_DIR)
FIGURE_DIR = ARTIFACT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)
figure_paths = []
registry_ids = [r['method_id'] for r in data['registry']]
styled_table(pd.DataFrame([{'Methods': len(registry_ids), 'Required method/lookahead combinations': 3 * len(registry_ids),
                           'Forecast cadence (days)': Parameters.forecast_days, 'Lookahead (days)': '7 / 30 / 90',
                           'Source-score checks': data['provenance'].get('source_score_audit', {}).get('observations', 546),
                           'Independent raw-label checks': audit_report['samples'], 'Raw targets match': audit_report['all_targets_match']}]),
             'Frozen experiment design and replication', 'Scoring recipes are deduplicated by complete transforms and missingness policy. '
             'Position sizing and position-count aliases do not multiply the forecast chart grid.')
styled_table(data['coverage'].query("period == '2026'"), 'Daily NAV target coverage',
             'The denominator includes all archived vaults and forecast dates, not only eligible or old vaults. '
             'CAGR/return and Sharpe use causally forward-filled daily NAV, retaining actual observation counts and carried-day counts. Administrative censoring means the lookahead has not finished.')
styled_table(raw_audit, 'Independent raw NAV label audit', 'A separate implementation recomputes endpoint return, log return, CAGR and Sharpe from raw share-price rows. '
             'Samples include flat, missing, delayed and price-guard divergence cases.', collapsed=True)
method_inventory = pd.DataFrame([{'ID': r['method_id'], 'Method': method_name(data, r['method_id']), 'Inputs': r['inputs'], 'Missing policy': r['missing_policy'],
                                 'Recipe': json.dumps(r['recipe'], sort_keys=True), 'Source': ', '.join(r['sources']), 'Aliases': '; '.join(r['aliases'])} for r in data['registry']])
styled_table(method_inventory, 'Complete scoring-method inventory', 'All implemented composite recipes from 04–07 are retained, including poor performers. '
             'Full recipes and coefficients are stored with the registry; identical aliases share one chart set.', collapsed=True)
mapping = data['provenance'].get('raw_mapping_audit', data['provenance'].get('mapping_audit', {}))
if mapping:
    styled_table(pd.DataFrame([mapping]), 'Price observation and candle mapping',
                 'Daily candles close on the last raw share-price observation inside the labelled day. Archive written_at is not a historical publication receipt. '
                 'Framework carried-price divergences remain visible and true tiny NAV losses are retained in forward targets.', collapsed=True)
universe_audit_path = ARTIFACT_DIR / 'universe-history-audit.csv'
if universe_audit_path.exists():
    styled_table(pd.read_csv(universe_audit_path), 'Frozen universe and historical coverage',
                 'The 514-vault parent population is kept for comparable experiments. It is a cached policy snapshot, not a historically complete vault population. '
                 'Older selected lists and the latest metadata show policy exclusions and newer candidates that the parent snapshot cannot cover.', collapsed=True)


Pinned score/label generation and predeclared policy rules — complete

### Frozen experiment design and replication — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Scoring recipes are deduplicated by complete transforms and missingness policy. Position sizing and position-count aliases do not multiply the forecast chart grid.

Methods,Required method/lookahead combinations,Forecast cadence (days),Lookahead (days),Source-score checks,Independent raw-label checks,Raw targets match
139,417,4,7 / 30 / 90,546,50,True


### Daily NAV target coverage — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

The denominator includes all archived vaults and forecast dates, not only eligible or old vaults. CAGR/return and Sharpe use causally forward-filled daily NAV, retaining actual observation counts and carried-day counts. Administrative censoring means the lookahead has not finished.

period,lookahead_days,scheduled_origins,matured_origins,last_matured_origin,complete_pairs,return_endpoint_pairs,sharpe_finite_pairs,endpoint_only_pairs,framework_approximate_pairs,vaults_with_complete_target,complete_fraction_all_vault_origins,administrative_censoring_fraction,flat_target_fraction_complete,low_variance_fraction_complete
2026,7,70,68,2026-09-26 00:00:00,26543,26543,19965,0,26543,511,73.77%,2.86%,24.78%,28.09%
2026,30,70,62,2026-09-02 00:00:00,23514,23514,19639,0,23514,493,65.35%,11.43%,16.48%,20.60%
2026,90,70,47,2026-07-04 00:00:00,16416,16416,14947,0,16416,451,45.63%,32.86%,8.95%,13.13%


...

ID,Method,Inputs,Missing policy,Recipe,Source,Aliases
v11,v11 (legacy 45d),2,propagate,"{""missing_policy"": ""propagate"", ""terms"": [{""cap"": 1.0, ""coefficient"": 0.6, ""days"": 360, ""drawdown_floor"": null, ""feature"": 40, ""metric"": ""CAGR"", ""period"": ""1y"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 3.0, ""coefficient"": 0.4, ""days"": 45, ""drawdown_floor"": null, ""feature"": 7, ""metric"": ""Sortino"", ""period"": ""45d compatibility"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}]}","06/07, 04 implemented search, 05 implemented",Legacy 45d; 04 evaluated 8baae743da0fb0bf532e117e; 05 Hard C1 S3; 05 Activity filter 10/45; 05 Hard C1 S3 / refill; 05 Feasible entrants; 05 Replacement hurdle 0.025; 05 Replacement hurdle 0.05; 05 Tie seed 11; 05 Tie seed 23; 05 Tie seed 37; 05 Tie seed 53; 05 Tie seed 71
score_917efb25281d,Legacy 45d / 3 inputs / 1m CAGR,3,propagate,"{""missing_policy"": ""propagate"", ""terms"": [{""cap"": 1.0, ""coefficient"": 0.2, ""days"": 30, ""drawdown_floor"": null, ""feature"": 0, ""metric"": ""CAGR"", ""period"": ""1m"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 1.0, ""coefficient"": 0.48, ""days"": 360, ""drawdown_floor"": null, ""feature"": 40, ""metric"": ""CAGR"", ""period"": ""1y"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 3.0, ""coefficient"": 0.32, ""days"": 45, ""drawdown_floor"": null, ""feature"": 7, ""metric"": ""Sortino"", ""period"": ""45d compatibility"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}]}",06/07,Legacy 45d / 3 inputs / 1m CAGR
score_d645792f5e24,Legacy 45d / 3 inputs / 2m CAGR,3,propagate,"{""missing_policy"": ""propagate"", ""terms"": [{""cap"": 1.0, ""coefficient"": 0.48, ""days"": 360, ""drawdown_floor"": null, ""feature"": 40, ""metric"": ""CAGR"", ""period"": ""1y"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 1.0, ""coefficient"": 0.2, ""days"": 60, ""drawdown_floor"": null, ""feature"": 15, ""metric"": ""CAGR"", ""period"": ""2m"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 3.0, ""coefficient"": 0.32, ""days"": 45, ""drawdown_floor"": null, ""feature"": 7, ""metric"": ""Sortino"", ""period"": ""45d compatibility"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}]}",06/07,Legacy 45d / 3 inputs / 2m CAGR
score_f9dbb810a719,Legacy 45d / 3 inputs / 3m CAGR,3,propagate,"{""missing_policy"": ""propagate"", ""terms"": [{""cap"": 1.0, ""coefficient"": 0.48, ""days"": 360, ""drawdown_floor"": null, ""feature"": 40, ""metric"": ""CAGR"", ""period"": ""1y"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 1.0, ""coefficient"": 0.2, ""days"": 90, ""drawdown_floor"": null, ""feature"": 20, ""metric"": ""CAGR"", ""period"": ""3m"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 3.0, ""coefficient"": 0.32, ""days"": 45, ""drawdown_floor"": null, ""feature"": 7, ""metric"": ""Sortino"", ""period"": ""45d compatibility"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}]}",06/07,Legacy 45d / 3 inputs / 3m CAGR
score_a7bdecffdc44,Legacy 45d / 3 inputs / 6m CAGR,3,propagate,"{""missing_policy"": ""propagate"", ""terms"": [{""cap"": 1.0, ""coefficient"": 0.2, ""days"": 180, ""drawdown_floor"": null, ""feature"": 30, ""metric"": ""CAGR"", ""period"": ""6m"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 1.0, ""coefficient"": 0.48, ""days"": 360, ""drawdown_floor"": null, ""feature"": 40, ""metric"": ""CAGR"", ""period"": ""1y"", ""ratio_floor"": false, ""risk_floor"": null, ""transform"": ""hard""}, {""cap"": 3.0, ""coefficient"": 0.32, ""days"": 45, ""drawdown_floor"": null, ""feature"": 7, ""metric"": ""Sortino"", ""period"": ""45d compatibility"", ""ratio_floor"": false, ""risk_floo

### Price observation and candle mapping — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Daily candles close on the last raw share-price observation inside the labelled day. Archive written_at is not a historical publication receipt. Framework carried-price divergences remain visible and true tiny NAV losses are retained in forward targets.

raw_observation_rows,daily_raw_rows,compared_candle_closes,close_mismatches,non_carried_close_mismatches,mismatch_vaults,max_close_absolute_error,clock_convention,publication_caveat,value_caveat
1513625,107321,106031,415,0,4,0.000,"Daily label D aggregates observations in [D,D+1d); close equals last raw share_price. Forecast T anchor close D=T-1d is complete at T.","written_at is archive update/write time, not an independently observed publication receipt. Publication history cannot be reconstructed.",share_price is cleaned per-share NAV; raw total_assets/total_supply diagnostics may differ because reconstruction removes cash flows.


### Frozen universe and historical coverage — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

The 514-vault parent population is kept for comparable experiments. It is a cached policy snapshot, not a historically complete vault population. Older selected lists and the latest metadata show policy exclusions and newer candidates that the parent snapshot cannot cover.

archive,generated_at,kind,vault_count,also_current_research_universe,absent_current_research_universe,current_not_in_prior,sha256
scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/inputs/vault-metadata.json,2026-10-07T18:43:20Z,published_metadata_all_hypercore,616,514,102,0,6c5ec12de41f1c10fc48e54a832eff065d14c77ea19c85fd482c46ea33a5382c
scratchpad/hyperliquid-vov-2nd/_artifacts-repaired-2026-10-07/inputs/vault-metadata.json,2026-10-07T09:51:29Z,published_metadata_all_hypercore,615,514,101,0,7201adaf27f925e9f0b4714c6c03c9c3831f3622e98ecd7e281ef1a845ddf532
scratchpad/hyperliquid-vov-2nd/_artifacts-repaired-2026-10-07/before-refresh/vault-metadata.json,2026-10-06T05:55:10Z,published_metadata_all_hypercore,611,513,98,1,2bd7e2e47cbde899fdb9278488ed1a49a9823c56187a1dd496f48cbeb542cd8a
scratchpad/hyperliquid-ic/_artifacts-allocation-decomposition/inputs/vault-metadata.json,2026-09-16T08:35:38Z,published_metadata_all_hypercore,603,511,92,3,1e9d6ac0ea3e8775f48eff5d8581ebec82baacdcb9a3eb123b4ae31e5ef13659
scratchpad/hyperliquid-ic/_artifacts-stability-screens/universe-membership.csv,—,earlier_selected_research_universe,359,343,16,171,83a20766f8175a84312bf7f1bdcfdda6207c0b7fe4c6d16cbdd46f5e5cc081f5
scratchpad/hyperliquid-ic/_artifacts-profitable-months-before-fee-fix/universe-membership.csv,—,earlier_selected_research_universe,360,344,16,170,ba2da798e15e322aa22d8f2b1ea80d6dce2092347e6d97070a3a0d14cdf5194b
scratchpad/hyperliquid-ic/_artifacts-quality-only/universe-membership.csv,—,earlier_selected_research_universe,359,343,16,171,83a20766f8175a84312bf7f1bdcfdda6207c0b7fe4c6d16cbdd46f5e5cc081f5
scratchpad/hyperliquid-ic/_artifacts-quality-only/leave-one-out/universe-membership.csv,—,earlier_selected_research_universe,359,343,16,171,83a20766f8175a84312bf7f1bdcfdda6207c0b7fe4c6d16cbdd46f5e5cc081f5
scratchpad/hyperliquid-ic/_artifacts-profitable-months/universe-membership.csv,—,earlier_selected_research_universe,360,344,16,170,ba2da798e15e322aa22d8f2b1ea80d6dce2092347e6d97070a3a0d14cdf5194b
scratchpad/hyperliquid-ic/_artifacts-monthly-calibration/universe-membership.csv,—,earlier_selected_research_universe,360,344,16,170,ba2da798e15e322aa22d8f2b1ea80d6dce2092347e6d97070a3a0d14cdf5194b


## Score versus future profitability: main scatter plots

For **v11, Blend 46d and the four-input candidate**, each **7-, 30- and 90-day lookahead** has three separate images, each with its own heading and description: **CAGR in the shared −100% to +300% linear range**, **CAGR over its full range**, and **Sharpe over its full range**. The zoomed view exposes ordinary outcomes; its overflow count refers to the separate tail audit. Scores and trends use all finite pairs. All charts use **2026-01-01 → 2026-10-05 (end exclusive, UTC)**; the complete appendix shows all remaining scoring methods.

In [2]:
chunk = render_method_appendix(data, FOCAL_METHODS, FIGURE_DIR)
scatter_manifest = chunk
figure_paths.extend(row['path'] for row in chunk)


### v11 (legacy 45d) · 7-day lookahead · CAGR: zoomed view — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. The shared linear display range is −100% to +300% CAGR. Counts above that range are printed on the plot; the separate full-range chart immediately below shows those observations. Raw-CAGR OLS is shown only in that full-range audit because extremes can dominate it. 

### v11 (legacy 45d) · 7-day lookahead · CAGR: full-range audit — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. This signed-log audit shows every finite annualised CAGR, including extreme tails. The dashed raw-CAGR OLS line can be dominated by a few extremes. 

### v11 (legacy 45d) · 7-day lookahead · Sharpe — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. A signed-log Sharpe axis expands the ordinary range and retains every extreme. The dashed line is descriptive pooled OLS. 

### v11 (legacy 45d) · 30-day lookahead · CAGR: zoomed view — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. The shared linear display range is −100% to +300% CAGR. Counts above that range are printed on the plot; the separate full-range chart immediately below shows those observations. Raw-CAGR OLS is shown only in that full-range audit because extremes can dominate it. 

### v11 (legacy 45d) · 30-day lookahead · CAGR: full-range audit — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. This signed-log audit shows every finite annualised CAGR, including extreme tails. The dashed raw-CAGR OLS line can be dominated by a few extremes. 

...

### Four inputs (180d CAGR + 30d Sortino) · 30-day lookahead · CAGR: full-range audit — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. This signed-log audit shows every finite annualised CAGR, including extreme tails. The dashed raw-CAGR OLS line can be dominated by a few extremes. 

### Four inputs (180d CAGR + 30d Sortino) · 30-day lookahead · Sharpe — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. A signed-log Sharpe axis expands the ordinary range and retains every extreme. The dashed line is descriptive pooled OLS. 

### Four inputs (180d CAGR + 30d Sortino) · 90-day lookahead · CAGR: zoomed view — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. The shared linear display range is −100% to +300% CAGR. Counts above that range are printed on the plot; the separate full-range chart immediately below shows those observations. Raw-CAGR OLS is shown only in that full-range audit because extremes can dominate it. Ninety-day 2026 relationships have too few independent windows for inferential claims.

### Four inputs (180d CAGR + 30d Sortino) · 90-day lookahead · CAGR: full-range audit — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. This signed-log audit shows every finite annualised CAGR, including extreme tails. The dashed raw-CAGR OLS line can be dominated by a few extremes. Ninety-day 2026 relationships have too few independent windows for inferential claims.

### Four inputs (180d CAGR + 30d Sortino) · 90-day lookahead · Sharpe — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

X is the frozen vault score; Y is the realised outcome on causally filled daily NAV. Each dot is one eligible vault/forecast date, coloured by origin quarter. Scores and all fitted trends use every finite pair; repeated and overlapping observations are not independent. Black is the robust trend; purple shows score-bin medians, with interquartile ranges in the zoomed view. A bin needs at least 20 pairs and five origins to appear. These ranges describe outcomes, not confidence intervals. A signed-log Sharpe axis expands the ordinary range and retains every extreme. The dashed line is descriptive pooled OLS. Ninety-day 2026 relationships have too few independent windows for inferential claims.

## Predictive relationships and calibration

Scores are rankings, not return-unit forecasts. Compare within-date relationships first, then chronological calibration and economic admission rules.

In [3]:
with compact_output('Shared forecast statistics — estimated 4–10 minutes'):
    statistics = analyse_data(data)
styled_table(focal_ic_table(data, statistics), 'Frozen controls: predictive rank relationships',
             'Mean IC weights forecast dates equally and ranks vaults available on that date. Positive means higher scores identify better future outcomes. '
             'Uncertainty uses whole date baskets; dashes indicate unsupported inference. Short-period 90-day confidence intervals are omitted.')
figure_paths.extend(overview_figures(data, statistics, FIGURE_DIR))


Shared forecast statistics — estimated 4–10 minutes — complete

### Frozen controls: predictive rank relationships — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Mean IC weights forecast dates equally and ranks vaults available on that date. Positive means higher scores identify better future outcomes. Uncertainty uses whole date baskets; dashes indicate unsupported inference. Short-period 90-day confidence intervals are omitted.

method,lookahead_days,outcome,mean_ic,ic_lower,ic_upper,supported_origins,distinct_vaults,paired_rows,inference_status
Blend 46d (two inputs),7,cagr,0.050,0.008,0.096,68,126,3141,conditional_exploratory_block_interval
Blend 46d (two inputs),7,sharpe,0.066,0.016,0.109,68,123,2849,conditional_exploratory_block_interval
Blend 46d (two inputs),30,cagr,0.087,-0.008,0.204,62,113,2728,conditional_exploratory_block_interval
Blend 46d (two inputs),30,sharpe,0.135,0.025,0.262,62,110,2626,conditional_exploratory_block_interval
Blend 46d (two inputs),90,cagr,0.219,—,—,47,87,1798,short_90_day_descriptive_only
Blend 46d (two inputs),90,sharpe,0.242,—,—,47,87,1788,short_90_day_descriptive_only
Four inputs (180d CAGR + 30d Sortino),7,cagr,0.058,0.011,0.110,68,126,3055,conditional_exploratory_block_interval
Four inputs (180d CAGR + 30d Sortino),7,sharpe,0.081,0.031,0.122,68,123,2813,conditional_exploratory_block_interval
Four inputs (180d CAGR + 30d Sortino),30,cagr,0.089,-0.005,0.198,62,113,2657,conditional_exploratory_block_interval
Four inputs (180d CAGR + 30d Sortino),30,sharpe,0.137,0.025,0.251,62,110,2582,conditional_exploratory_block_interval


### Predictive relationships across methods and horizons — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Equal-date mean Spearman correlation within eligible candidates: positive means higher scores ranked better future outcomes. Rows follow the frozen registry, not a winner sort. Dashes indicate insufficient support. The shared colour range is −0.30 to +0.30 so small differences remain visible; starred cells exceed the colour range but retain their true numbers. The 90-day short-history column is descriptive only; overlapping dots are not independent samples.

### Predictive relationships across methods and horizons — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Equal-date mean Spearman correlation within eligible candidates: positive means higher scores ranked better future outcomes. Rows follow the frozen registry, not a winner sort. Dashes indicate insufficient support. The shared colour range is −0.30 to +0.30 so small differences remain visible; starred cells exceed the colour range but retain their true numbers. The 90-day short-history column is descriptive only; overlapping dots are not independent samples.

...

### Score calibration and tail risk: return q05 — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fixed score bins retain ties. Median return, loss frequency and the lower tail answer different questions. Curves are retrospective diagnostics, not fitted live forecasts; counts and attrition are shown separately.

### Candidate rank: median return — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare the upper candidate ranks with lower-ranked candidates. Candidate ranks use average ranks for ties; held-position hysteresis means these are not necessarily the actual marginal entries.

### Candidate rank: loss probability — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare the upper candidate ranks with lower-ranked candidates. Candidate ranks use average ranks for ties; held-position hysteresis means these are not necessarily the actual marginal entries.

### Entry timing and stale-mark sensitivity — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Keep the score fixed while delaying the NAV anchor one or two days. A disappearing relationship would suggest dependence on shared endpoints or timing, rather than persistent forward information.

### How long does the score information persist? — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Additional horizons diagnose information decay; they do not replace the required 7/30/90-day charts. Long horizons have fewer matured origins, so this is not a matched-sample comparison.

In [4]:
figure_paths.extend(display_calibration(data, statistics, FIGURE_DIR))
for frame_name, title, description in [
    ('coverage', 'Eligible score coverage and missing cohorts', 'Separate missing scores, valid zeros, unsupported future prices and undefined ratios; coverage differences are not forecasting skill.'),
    ('clipping', 'Score plateaus and ties', 'Hard clips can turn materially different vaults into equal scores. This audit counts zero/one scores and within-date ties.'),
    ('robustness', 'Vault, quarter, variance and attrition sensitivities', 'Compare omissions and target-quality sensitivities with the same frozen method; no worst observation is deleted from the primary estimate.'),
    ('lagged_score', 'Freshness placebo', 'A stale score predicts the same future outcome. Requested lag 30 days maps to the last available four-day score, while 60 days is exact.'),
    ('incremental_information', 'Incremental information from the extra inputs', 'Residualise the focal four-input score against its two-input core within each date; this tests information beyond the shared core, not a new optimised score.')]:
    frame = statistics.get(frame_name, pd.DataFrame())
    if not frame.empty:
        if 'period' in frame:
            frame = frame[frame.period.eq('2026')]
        styled_table(frame, title, description, collapsed=True)


### 30-day chronological forecast errors — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Monthly models use only earlier labels whose full future windows had become observable. Main estimates require at least 20 actual training origins and 10 vaults for the specific outcome. Lower MAE/MSE/Brier is better; outcome units differ. Forecast dates overlap; the support table reports training and evaluation counts.

method,lookahead_days,outcome,model,equal_date_mae,equal_date_mse,brier_score,origin_count,vault_count
Blend 46d (two inputs),30,log_return,constant,0.142,0.094,—,62,113
Blend 46d (two inputs),30,log_return,isotonic,0.154,0.094,—,62,113
Blend 46d (two inputs),30,log_return,linear,0.147,0.093,—,62,113
Blend 46d (two inputs),30,loss,constant,0.500,0.251,0.251,62,113
Blend 46d (two inputs),30,loss,isotonic,0.496,0.249,0.249,62,113
Blend 46d (two inputs),30,loss,linear,0.504,0.256,0.256,62,113
Blend 46d (two inputs),30,sharpe,constant,3.324,16.130,—,62,110
Blend 46d (two inputs),30,sharpe,isotonic,3.278,15.862,—,62,110
Blend 46d (two inputs),30,sharpe,linear,3.322,16.117,—,62,110
Four inputs (180d CAGR + 30d Sortino),30,log_return,constant,0.144,0.098,—,62,113


### 7-day chronological forecast errors — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Monthly models use only earlier labels whose full future windows had become observable. Main estimates require at least 20 actual training origins and 10 vaults for the specific outcome. Lower MAE/MSE/Brier is better; outcome units differ. Forecast dates overlap; the support table reports training and evaluation counts.

method,lookahead_days,outcome,model,equal_date_mae,equal_date_mse,brier_score,origin_count,vault_count
Blend 46d (two inputs),7,log_return,constant,0.053,0.014,—,68,126
Blend 46d (two inputs),7,log_return,isotonic,0.057,0.014,—,68,126
Blend 46d (two inputs),7,log_return,linear,0.054,0.014,—,68,126
Blend 46d (two inputs),7,loss,constant,0.484,0.264,0.264,68,126
Blend 46d (two inputs),7,loss,isotonic,0.486,0.263,0.263,68,126
Blend 46d (two inputs),7,loss,linear,0.484,0.263,0.263,68,126
Blend 46d (two inputs),7,sharpe,constant,6.866,69.024,—,68,123
Blend 46d (two inputs),7,sharpe,isotonic,6.860,69.093,—,68,123
Blend 46d (two inputs),7,sharpe,linear,6.869,69.073,—,68,123
Four inputs (180d CAGR + 30d Sortino),7,log_return,constant,0.054,0.015,—,68,126


### 90-day chronological forecast errors — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Monthly models use only earlier labels whose full future windows had become observable. Main estimates require at least 20 actual training origins and 10 vaults for the specific outcome. Lower MAE/MSE/Brier is better; outcome units differ. Forecast dates overlap; the support table reports training and evaluation counts.

...

sensitivity,excluded,outcome,mean_ic,baseline_ic,difference,supported_origins,method_id,lookahead_days,period,population
leave_quarter_out,2026Q1,cagr,0.012,0.050,-0.038,45,v11,7,2026,eligible
leave_quarter_out,2026Q2,cagr,0.062,0.050,0.012,45,v11,7,2026,eligible
leave_quarter_out,2026Q3,cagr,0.076,0.050,0.026,46,v11,7,2026,eligible
leave_quarter_out,2026Q4,cagr,0.050,0.050,0.000,68,v11,7,2026,eligible
leave_vault_out,268556798,cagr,0.050,0.050,0.000,68,v11,7,2026,eligible
leave_vault_out,268585277,cagr,0.051,0.050,0.001,68,v11,7,2026,eligible
leave_vault_out,268652292,cagr,0.049,0.050,-0.001,68,v11,7,2026,eligible
leave_vault_out,268659860,cagr,0.050,0.050,0.000,68,v11,7,2026,eligible
leave_vault_out,268724381,cagr,0.052,0.050,0.002,68,v11,7,2026,eligible
leave_vault_out,268745679,cagr,0.050,0.050,-0.000,68,v11,7,2026,eligible


### Freshness placebo — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

A stale score predicts the same future outcome. Requested lag 30 days maps to the last available four-day score, while 60 days is exact.

method_id,lookahead_days,period,requested_lag_days,actual_lag_days,lag_reason,mean_ic,median_ic,positive_date_fraction,supported_origins,paired_origins,mean_paired_vaults,mean_tie_fraction,ic_lower,ic_upper,hac_se,effective_origins,minimum_detectable_ic,nonoverlap_mean_ic,nonoverlap_origins,independent_blocks,block_origins,inference_status
v11,30,2026,30,32,last available four-day score at or before requested lag,0.066,0.111,69.35%,62,62,37.532,38.77%,-0.047,0.224,0.073,11.406,0.204,0.111,8,7,8,conditional_exploratory_block_interval
v11,30,2026,60,60,last available four-day score at or before requested lag,0.082,0.131,64.52%,62,62,31.855,38.51%,-0.026,0.221,0.069,12.171,0.195,0.136,8,7,8,conditional_exploratory_block_interval
blend_46d,30,2026,30,32,last available four-day score at or before requested lag,0.076,0.086,67.74%,62,62,37.565,33.86%,-0.035,0.231,0.073,11.306,0.205,0.118,8,7,8,conditional_exploratory_block_interval
blend_46d,30,2026,60,60,last available four-day score at or before requested lag,0.101,0.152,69.35%,62,62,31.871,34.94%,-0.007,0.242,0.070,12.124,0.196,0.151,8,7,8,conditional_exploratory_block_interval
focal_four,30,2026,30,32,last available four-day score at or before requested lag,0.069,0.110,70.97%,62,62,36.565,29.37%,-0.050,0.219,0.073,11.197,0.203,0.138,8,6,9,conditional_exploratory_block_interval
focal_four,30,2026,60,60,last available four-day score at or before requested lag,0.071,0.151,62.90%,62,62,30.758,28.92%,-0.041,0.218,0.072,12.699,0.201,0.129,8,7,8,conditional_exploratory_block_interval


### Incremental information from the extra inputs — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Residualise the focal four-input score against its two-input core within each date; this tests information beyond the shared core, not a new optimised score. Showing the first 36 of 213 rows to keep the notebook renderable; the complete analysis remains in its cached dataframe.

origin_at,outcome,partial_rank_ic,vault_count,method_id,core_method_id,status
2025-06-15,log_return,0.705,10,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-06-15,sharpe,0.754,10,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-06-19,log_return,0.100,11,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-06-19,sharpe,0.288,11,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-06-23,log_return,-0.082,11,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-06-23,sharpe,0.041,11,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-06-27,log_return,-0.438,12,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-06-27,sharpe,-0.470,12,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-07-01,log_return,0.023,11,focal_four,v11,descriptive_linear_core_residual_not_causal
2025-07-01,sharpe,-0.145,11,focal_four,v11,descriptive_linear_core_residual_not_causal


## Interactive focal views

These zoomable 30-day views supplement the complete static appendix. Raw CAGR units can be dominated by extreme annualisation; the appendix uses signed-log axes without dropping points.

In [5]:
for method_id in FOCAL_METHODS:
    frame = join_method(data, method_id, 30)
    for target in ('future_cagr', 'future_sharpe'):
        label = 'CAGR' if target == 'future_cagr' else 'Sharpe'
        intro(method_name(data, method_id) + ' · interactive 30-day ' + label,
              'Each point is one vault/forecast date, coloured by quarter. The robust line is descriptive. '
              'CAGR starts in a zoomed range; double-click or zoom out for all extremes. No target values are clipped.')
        fig = interactive_scatter(data, method_id, 30, frame, target=target)
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))


### v11 (legacy 45d) · interactive 30-day CAGR — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one vault/forecast date, coloured by quarter. The robust line is descriptive. CAGR starts in a zoomed range; double-click or zoom out for all extremes. No target values are clipped.

### v11 (legacy 45d) · interactive 30-day Sharpe — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one vault/forecast date, coloured by quarter. The robust line is descriptive. CAGR starts in a zoomed range; double-click or zoom out for all extremes. No target values are clipped.

### Blend 46d (two inputs) · interactive 30-day CAGR — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one vault/forecast date, coloured by quarter. The robust line is descriptive. CAGR starts in a zoomed range; double-click or zoom out for all extremes. No target values are clipped.

### Blend 46d (two inputs) · interactive 30-day Sharpe — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one vault/forecast date, coloured by quarter. The robust line is descriptive. CAGR starts in a zoomed range; double-click or zoom out for all extremes. No target values are clipped.

### Four inputs (180d CAGR + 30d Sortino) · interactive 30-day CAGR — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one vault/forecast date, coloured by quarter. The robust line is descriptive. CAGR starts in a zoomed range; double-click or zoom out for all extremes. No target values are clipped.

### Four inputs (180d CAGR + 30d Sortino) · interactive 30-day Sharpe — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one vault/forecast date, coloured by quarter. The robust line is descriptive. CAGR starts in a zoomed range; double-click or zoom out for all extremes. No target values are clipped.

## Executable entry-floor experiments

The forecast cadence remains four days; these policies trade on the original two-day schedule. No scoring coefficients or selector ranks are retuned. Admission floors veto selected new entrants, so the source may still exit a displaced incumbent; no hidden replacement entrant is added.

In [6]:
from tqdm_loggable.auto import tqdm
with compact_output('Source-controlled threshold replays — estimated 10–20 minutes'):
    progress_bar = tqdm(total=132, desc='Entry floors and matched controls')
    def execution_progress(done, total, row):
        progress_bar.total = total
        progress_bar.set_postfix(period=row['Period'], method=row['Method'], n=row['Positions'], policy=row['Threshold policy'], refresh=False)
        progress_bar.update(done - progress_bar.n)
    try:
        execution = run_threshold_experiments(data, statistics, include_random=True, progress=execution_progress)
    finally:
        progress_bar.close()
from audit_sparse_metrics import refresh_foregone_targets
execution = refresh_foregone_targets(data, execution)
figure_paths.extend(display_execution(data, execution, FIGURE_DIR))


Source-controlled threshold replays — estimated 10–20 minutes — complete

### Distinct net portfolio performance side by side — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fresh $150,000 replays keep the source two-day cadence, 90-day inverse variance, caps, guards, hysteresis and estimated costs. Slot-to-cash preserves the surviving proposed dollar targets; renormalisation redistributes among the remaining selected holdings. A no-floor-equivalent row rejected no entries and is not a new successful strategy.

Method,Positions,Threshold policy,Treatment,CAGR,Sharpe,Maximum drawdown,Mean cash weight,Mean vaults held,Trades,Rejected entry proposals,Rejected entry dollars (USD),Floor-active decision fraction,No-floor equivalent
v11,6,none,slot_to_cash,41.20%,1.960,-6.40%,8.83%,6.000,581,0,$0,0.00%,True
v11,7,none,slot_to_cash,33.47%,1.746,-5.11%,5.09%,6.978,688,0,$0,0.00%,True
focal_four,6,none,slot_to_cash,51.77%,2.303,-6.35%,10.18%,5.993,569,0,$0,0.00%,True
focal_four,7,none,slot_to_cash,30.85%,1.582,-6.42%,6.21%,6.957,726,0,$0,0.00%,True
focal_four,7,percentile_75,slot_to_cash,31.39%,1.605,-6.44%,6.26%,6.949,725,1,"$5,670",100.00%,False
focal_four,7,percentile_75,renormalise_within_caps,31.35%,1.603,-6.43%,6.20%,6.949,725,1,"$5,670",100.00%,False
blend_46d,6,none,slot_to_cash,41.38%,2.119,-4.85%,7.43%,6.000,583,0,$0,0.00%,True
blend_46d,7,none,slot_to_cash,47.93%,2.459,-4.71%,5.03%,6.978,663,0,$0,0.00%,True


### Complete primary policy arms — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

All 30 non-random arms are retained, including verified no-entry-change duplicates. The compact comparison above shows only baseline and binding policies.

Method,Positions,Threshold policy,Treatment,CAGR,Sharpe,Maximum drawdown,Mean cash weight,Mean vaults held,Trades,Rejected entry proposals,Rejected entry dollars (USD),Floor-active decision fraction,No-floor equivalent
v11,6,none,slot_to_cash,41.20%,1.960,-6.40%,8.83%,6.000,581,0,$0,0.00%,True
v11,6,monthly_absolute,slot_to_cash,41.20%,1.960,-6.40%,8.83%,6.000,581,0,$0,88.49%,True
v11,6,monthly_absolute,renormalise_within_caps,41.20%,1.960,-6.40%,8.83%,6.000,581,0,$0,88.49%,True
v11,6,percentile_75,slot_to_cash,41.20%,1.960,-6.40%,8.83%,6.000,581,0,$0,100.00%,True
v11,6,percentile_75,renormalise_within_caps,41.20%,1.960,-6.40%,8.83%,6.000,581,0,$0,100.00%,True
v11,7,none,slot_to_cash,33.47%,1.746,-5.11%,5.09%,6.978,688,0,$0,0.00%,True
v11,7,monthly_absolute,slot_to_cash,33.47%,1.746,-5.11%,5.09%,6.978,688,0,$0,88.49%,True
v11,7,monthly_absolute,renormalise_within_caps,33.47%,1.746,-5.11%,5.09%,6.978,688,0,$0,88.49%,True
v11,7,percentile_75,slot_to_cash,33.47%,1.746,-5.11%,5.09%,6.978,688,0,$0,100.00%,True
v11,7,percentile_75,renormalise_within_caps,33.47%,1.746,-5.11%,5.09%,6.978,688,0,$0,100.00%,True


...

### Portfolio profit heatmap by quarter — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Quarterly compounded net portfolio returns use actual replay equity. The final quarter is partial. No-entry-change duplicates are omitted; this heatmap is different from quarter-coloured forward labels.

### Secondary full-history portfolio comparison — 2023-03-01 → 2026-10-05 (end exclusive, UTC)

Identical policy rules over the longer history. This period overlaps 2026 and was already used in earlier research, so it is not independent confirmation.

Method,Positions,Threshold policy,Treatment,CAGR,Sharpe,Maximum drawdown,Mean cash weight,Mean vaults held,Trades,Rejected entry proposals,Rejected entry dollars (USD),Floor-active decision fraction,No-floor equivalent
v11,6,none,slot_to_cash,15.77%,1.238,-7.92%,41.81%,4.324,1074,0,$0,0.00%,True
v11,6,monthly_absolute,slot_to_cash,15.77%,1.238,-7.92%,41.81%,4.324,1074,0,$0,20.10%,True
v11,6,monthly_absolute,renormalise_within_caps,15.77%,1.238,-7.92%,41.81%,4.324,1074,0,$0,20.10%,True
v11,6,percentile_75,slot_to_cash,16.24%,1.266,-7.92%,43.69%,4.087,1020,145,"$1,852,742",70.59%,False
v11,6,percentile_75,renormalise_within_caps,16.24%,1.262,-7.92%,43.33%,4.087,1014,145,"$1,845,242",70.59%,False
v11,7,none,slot_to_cash,15.31%,1.154,-7.92%,38.44%,4.830,1231,0,$0,0.00%,True
v11,7,monthly_absolute,slot_to_cash,15.31%,1.154,-7.92%,38.44%,4.830,1231,0,$0,20.10%,True
v11,7,monthly_absolute,renormalise_within_caps,15.31%,1.154,-7.92%,38.44%,4.830,1231,0,$0,20.10%,True
v11,7,percentile_75,slot_to_cash,16.14%,1.224,-7.92%,40.40%,4.548,1166,185,"$2,502,097",70.59%,False
v11,7,percentile_75,renormalise_within_caps,15.62%,1.169,-7.92%,39.44%,4.548,1145,185,"$2,466,189",70.59%,False


### Assumed cash-yield sensitivity — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Zero yield is the actual source replay. The 3%/5% annual scenarios add hypothetical daily yield to prior cash weights on a fixed ledger, without changing sizing or fees. They are attribution scenarios, not executable cash-sweep backtests.

Method,Positions,Assumed annual cash yield,Fixed-ledger CAGR,Fixed-ledger Sharpe,Fixed-ledger maximum drawdown
v11,6,0.00%,41.20%,1.960,-6.40%
v11,6,3.00%,41.58%,1.974,-6.36%
v11,6,5.00%,41.82%,1.983,-6.32%
v11,7,0.00%,33.47%,1.746,-5.11%
v11,7,3.00%,33.68%,1.755,-5.08%
v11,7,5.00%,33.82%,1.761,-5.08%
focal_four,6,0.00%,51.77%,2.303,-6.35%
focal_four,6,3.00%,52.23%,2.320,-6.33%
focal_four,6,5.00%,52.53%,2.330,-6.31%
focal_four,7,0.00%,30.85%,1.582,-6.42%


In [7]:
for frame_name, title, description in [
    ('jumps', 'Strongest booked portfolio moves', 'Inspect the leading position behind large positive and negative equity changes; concentrated vault events can dominate performance.'),
    ('positions', 'Individual positions and curator flags', 'Realised position profits and losses retain curator exclusion/quarantine flags. The flags are a diagnostic, not a hindsight removal.'),
    ('rejected_outcomes', 'What rejected entries subsequently earned', 'These are static gross 30-day outcomes from proposed entries using causal daily NAV carry, not counterfactual strategy PnL. Actual endpoint counts remain separate; prelaunch and terminally censored futures stay missing.')]:
    frame = execution.get(frame_name, pd.DataFrame())
    if not frame.empty:
        frame = frame[frame.Period.eq('2026') & frame.Seed.isna()]
        if frame_name == 'positions':
            frame = pd.concat([frame.nlargest(12, 'Final profit (USD)'), frame.nsmallest(12, 'Final profit (USD)')]).drop_duplicates()
        styled_table(frame, title, description, collapsed=True)


### Strongest booked portfolio moves — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Inspect the leading position behind large positive and negative equity changes; concentrated vault events can dominate performance. Showing the first 36 of 150 rows to keep the notebook renderable; the complete analysis remains in its cached dataframe.

Date,equity,daily_return,cash_weight,previous_valuation_at,Leading position vault,Leading position address,Leading position PnL (USD),Leading position curator excluded,Leading position quarantined at event,Leader share of same-sign day PnL,Leader contribution sign,BTCUSDT matching interval return,Reference candle start,Reference candle end,ETHUSDT matching interval return,Period,Method,Positions,Threshold policy,Treatment,Seed
2026-06-26,177462.884,6.89%,23.43%,2026-06-24,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$6,930",False,False,0.584,positive,$-0,2026-06-23,2026-06-25,$-0,2026,v11,6,none,slot_to_cash,—
2026-09-30,191190.164,-4.86%,58.23%,2026-09-28,DOEZOE,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,"$-8,856",False,False,0.903,negative,$-0,2026-09-27,2026-09-29,$-0,2026,v11,6,none,slot_to_cash,—
2026-06-24,166028.400,4.11%,25.07%,2026-06-22,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$8,969",False,False,1.000,positive,$-0,2026-06-21,2026-06-23,$-0,2026,v11,6,none,slot_to_cash,—
2026-07-08,193699.245,3.66%,4.69%,2026-07-06,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$6,208",False,False,0.871,positive,$-0,2026-07-05,2026-07-07,$-0,2026,v11,6,none,slot_to_cash,—
2026-07-04,186623.184,3.30%,26.65%,2026-07-02,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$5,228",False,False,0.754,positive,$0,2026-07-01,2026-07-03,$0,2026,v11,6,none,slot_to_cash,—
2026-06-26,177462.884,6.89%,23.43%,2026-06-24,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$6,930",False,False,0.584,positive,$-0,2026-06-23,2026-06-25,$-0,2026,v11,6,monthly_absolute,slot_to_cash,—
2026-09-30,191190.164,-4.86%,58.23%,2026-09-28,DOEZOE,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,"$-8,856",False,False,0.903,negative,$-0,2026-09-27,2026-09-29,$-0,2026,v11,6,monthly_absolute,slot_to_cash,—
2026-06-24,166028.400,4.11%,25.07%,2026-06-22,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$8,969",False,False,1.000,positive,$-0,2026-06-21,2026-06-23,$-0,2026,v11,6,monthly_absolute,slot_to_cash,—
2026-07-08,193699.245,3.66%,4.69%,2026-07-06,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$6,208",False,False,0.871,positive,$-0,2026-07-05,2026-07-07,$-0,2026,v11,6,monthly_absolute,slot_to_cash,—
2026-07-04,186623.184,3.30%,26.65%,2026-07-02,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,"$5,228",False,False,0.754,positive,$0,2026-07-01,2026-07-03,$0,2026,v11,6,monthly_absolute,slot_to_cash,—


### Individual positions and curator flags — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Realised position profits and losses retain curator exclusion/quarantine flags. The flags are a diagnostic, not a hindsight removal.

Position id,Vault,Address,Opened,Closed,Final profit (USD),Curator excluded,Quarantined at entry,Period,Method,Positions,Threshold policy,Treatment,Seed
67,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$26,168",False,False,2026,focal_four,6,none,slot_to_cash,—
67,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$26,168",False,False,2026,focal_four,6,monthly_absolute,slot_to_cash,—
67,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$26,168",False,False,2026,focal_four,6,monthly_absolute,renormalise_within_caps,—
67,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$26,168",False,False,2026,focal_four,6,percentile_75,slot_to_cash,—
67,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$26,168",False,False,2026,focal_four,6,percentile_75,renormalise_within_caps,—
64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$23,285",False,False,2026,v11,6,none,slot_to_cash,—
64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$23,285",False,False,2026,v11,6,monthly_absolute,slot_to_cash,—
64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$23,285",False,False,2026,v11,6,monthly_absolute,renormalise_within_caps,—
64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$23,285",False,False,2026,v11,6,percentile_75,slot_to_cash,—
64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,"$23,285",False,False,2026,v11,6,percentile_75,renormalise_within_caps,—


### What rejected entries subsequently earned — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

These are static gross 30-day outcomes from proposed entries using causal daily NAV carry, not counterfactual strategy PnL. Actual endpoint counts remain separate; prelaunch and terminally censored futures stay missing.

Period,Method,Positions,Threshold policy,Treatment,Seed,Rejected proposals,Matured foregone labels,Carried endpoint labels,"Foregone 30d gross return, unweighted",Foregone 30d gross marked PnL (USD)
2026,focal_four,7,percentile_75,slot_to_cash,—,1,1,1,-37.79%,"$-2,143"
2026,focal_four,7,percentile_75,renormalise_within_caps,—,1,1,1,-37.79%,"$-2,143"


## Missing-score entry sensitivity

An unscored vault is distinct from a valid zero score. This diagnostic keeps the source selector and incumbent rules, vetoes only new unscored entries, and leaves their proposed dollars as cash.

In [8]:
from score_forecast_missing import run_missing_sensitivity
with compact_output('Missing-score entry sensitivity — six historical replays'):
    missing_bundle = run_missing_sensitivity(data, execution)
missing_sensitivity = missing_bundle['summary']
missing_columns = ['Method', 'Positions', 'CAGR', 'Sharpe', 'Maximum drawdown', 'Mean cash weight', 'Rejected entry proposals', 'CAGR delta vs no-floor', 'Sharpe delta vs no-floor', 'Derived identical']
styled_table(missing_sensitivity[missing_sensitivity.Period.eq('2026')][missing_columns], 'Unscored admission in the short period',
             'The six baseline ledgers contain no funded unscored new entries, so the identical entry-veto paths are reused after explicit checks. These are equivalent policies, not additional independent successes.')
styled_table(missing_sensitivity[missing_sensitivity.Period.eq('Long')][missing_columns], 'Unscored admission in earlier history',
             'Six fresh full-history engine replays veto only nonfinite-score new entries. Finite zero scores, existing holdings, sizing, guards, fees and selection machinery retain the source rules.', period=LONG_PERIOD, collapsed=True)


Missing-score entry sensitivity — six historical replays — complete

### Unscored admission in the short period — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

The six baseline ledgers contain no funded unscored new entries, so the identical entry-veto paths are reused after explicit checks. These are equivalent policies, not additional independent successes.

Method,Positions,CAGR,Sharpe,Maximum drawdown,Mean cash weight,Rejected entry proposals,CAGR delta vs no-floor,Sharpe delta vs no-floor,Derived identical
v11,6,41.20%,1.960,-6.40%,8.83%,0,0.00%,0.000,True
v11,7,33.47%,1.746,-5.11%,5.09%,0,0.00%,0.000,True
blend_46d,6,41.38%,2.119,-4.85%,7.43%,0,0.00%,0.000,True
blend_46d,7,47.93%,2.459,-4.71%,5.03%,0,0.00%,0.000,True
focal_four,6,51.77%,2.303,-6.35%,10.18%,0,0.00%,0.000,True
focal_four,7,30.85%,1.582,-6.42%,6.21%,0,0.00%,0.000,True


### Unscored admission in earlier history — 2023-03-01 → 2026-10-05 (end exclusive, UTC)

Six fresh full-history engine replays veto only nonfinite-score new entries. Finite zero scores, existing holdings, sizing, guards, fees and selection machinery retain the source rules.

Method,Positions,CAGR,Sharpe,Maximum drawdown,Mean cash weight,Rejected entry proposals,CAGR delta vs no-floor,Sharpe delta vs no-floor,Derived identical
v11,6,11.11%,0.971,-6.39%,59.23%,837,-4.66%,-0.267,False
v11,7,10.41%,0.923,-7.21%,58.10%,972,-4.90%,-0.231,False
blend_46d,6,10.62%,0.967,-7.78%,58.62%,834,-4.79%,-0.268,False
blend_46d,7,13.09%,1.180,-7.39%,58.03%,969,-5.31%,-0.210,False
focal_four,6,18.30%,1.541,-6.29%,61.95%,874,-3.75%,-0.077,False
focal_four,7,12.37%,1.048,-9.78%,60.56%,1007,-4.93%,-0.214,False


## Population, price-path and market-exposure checks

These supplementary analyses test whether apparent predictability depends on the eligible population, carried price marks or broad market exposure. They do not change the frozen scoring rules.

In [9]:
from score_forecast_extensions import run_extensions
with compact_output('Population and market-exposure sensitivities — estimated 2–5 minutes'):
    extensions = run_extensions(data, statistics)
for name, frame in extensions.items():
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        shown = frame[frame.period.eq('2026')] if 'period' in frame else frame
        if shown.empty:
            continue
        descriptions = extensions.get('descriptions', {})
        styled_table(shown, name.replace('_', ' ').capitalize(), descriptions.get(name,
                     'Supporting diagnostics use frozen methods and causal input cutoffs; unsupported estimates remain missing.'), collapsed=True)


Population and market-exposure sensitivities — estimated 2–5 minutes — complete

### Cohort summary — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Same frozen scores across eligible, observed, common-score, guard, age and activity cohorts. Guard comparisons use current policy retrospectively within the fixed 514-vault parent universe; they do not recover excluded historical vaults. Showing the first 36 of 162 rows to keep the notebook renderable; the complete analysis remains in its cached dataframe.

method_id,lookahead_days,period,cohort,outcome,historical_policy_verified,rows,paired_rows,paired_vaults,origin_count,mean_ic,median_ic,positive_date_fraction,supported_origins,paired_origins,mean_paired_vaults,mean_tie_fraction,ic_lower,ic_upper,hac_se,effective_origins,minimum_detectable_ic,nonoverlap_mean_ic,nonoverlap_origins,independent_blocks,block_origins,inference_status
v11,7,2026,all_observed_parent_universe,return,False,35980,5812,160,70,0.038,0.044,61.76%,68,68.000,85.471,52.10%,0.002,0.080,0.021,60.639,0.058,0.067,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,all_observed_parent_universe,sharpe,False,35980,5031,149,70,0.072,0.062,69.12%,68,68.000,73.985,50.03%,0.031,0.112,0.022,49.349,0.061,0.090,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,source_compatible_eligible,return,False,12702,3137,126,70,0.050,0.062,55.88%,68,68.000,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,source_compatible_eligible,sharpe,False,12702,2848,123,70,0.063,0.084,69.12%,68,68.000,41.882,35.55%,0.013,0.107,0.025,43.344,0.071,0.086,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,eligible_common_focal_finite_score,return,False,3188,3055,126,70,0.048,0.078,58.82%,68,68.000,44.926,36.47%,0.003,0.097,0.027,45.826,0.075,0.078,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,eligible_common_focal_finite_score,sharpe,False,3188,2813,123,70,0.066,0.089,69.12%,68,68.000,41.368,35.82%,0.015,0.111,0.026,43.237,0.072,0.088,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,current_guard_allows_only_retrospective,return,False,35980,5812,160,70,0.038,0.044,61.76%,68,68.000,85.471,52.10%,0.002,0.080,0.021,60.639,0.058,0.067,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,current_guard_allows_only_retrospective,sharpe,False,35980,5031,149,70,0.072,0.062,69.12%,68,68.000,73.985,50.03%,0.031,0.112,0.022,49.349,0.061,0.090,34.000,34.000,2.000,conditional_exploratory_block_interval
v11,7,2026,current_guard_denies_only_retrospective,return,False,0,0,0,0,—,—,—,0,—,—,—,—,—,—,—,—,—,—,—,—,empty_cohort
v11,7,2026,current_guard_denies_only_retrospective,sharpe,False,0,0,0,0,—,—,—,0,—,—,—,—,—,—,—,—,—,—,—,—,empty_cohort


### Target basis summary — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Causally filled raw NAV targets versus framework guarded/filled prices. Both retain held-NAV assumptions; carried prices are not extra actual observations. Showing the first 36 of 72 rows to keep the notebook renderable; the complete analysis remains in its cached dataframe.

method_id,lookahead_days,period,report_period,basis,outcome,rows,paired_rows,paired_vaults,origin_count,mean_ic,median_ic,positive_date_fraction,supported_origins,paired_origins,mean_paired_vaults,mean_tie_fraction,ic_lower,ic_upper,hac_se,effective_origins,minimum_detectable_ic,nonoverlap_mean_ic,nonoverlap_origins,independent_blocks,block_origins,inference_status
v11,7,2026,2026,causally_filled_raw_own_valid,return,12702,3137,126,70,0.050,0.062,55.88%,68,68,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34,34,2,conditional_exploratory_block_interval
v11,7,2026,2026,framework_filled_own_valid_approximation,return,12702,3137,126,70,0.050,0.062,55.88%,68,68,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34,34,2,conditional_exploratory_block_interval
v11,7,2026,2026,causally_filled_raw_common_valid,return,12370,3137,126,68,0.050,0.062,55.88%,68,68,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34,34,2,conditional_exploratory_block_interval
v11,7,2026,2026,framework_filled_common_valid_approximation,return,12370,3137,126,68,0.050,0.062,55.88%,68,68,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34,34,2,conditional_exploratory_block_interval
v11,7,2026,2026,causally_filled_raw_own_valid,sharpe,12702,2848,123,70,0.063,0.084,69.12%,68,68,41.882,35.55%,0.013,0.107,0.025,43.344,0.071,0.086,34,34,2,conditional_exploratory_block_interval
v11,7,2026,2026,framework_filled_own_valid_approximation,sharpe,12702,2848,123,70,0.063,0.084,69.12%,68,68,41.882,35.55%,0.013,0.107,0.025,43.344,0.071,0.086,34,34,2,conditional_exploratory_block_interval
v11,7,2026,2026,causally_filled_raw_common_valid,sharpe,10818,2848,123,68,0.063,0.084,69.12%,68,68,41.882,35.55%,0.013,0.107,0.025,43.344,0.071,0.086,34,34,2,conditional_exploratory_block_interval
v11,7,2026,2026,framework_filled_common_valid_approximation,sharpe,10818,2848,123,68,0.063,0.084,69.12%,68,68,41.882,35.55%,0.013,0.107,0.025,43.344,0.071,0.086,34,34,2,conditional_exploratory_block_interval
v11,30,2026,2026,causally_filled_raw_own_valid,return,12702,2726,113,70,0.088,0.138,67.74%,62,62,43.968,36.20%,-0.007,0.200,0.068,8.371,0.191,0.075,8,3,20,conditional_exploratory_block_interval
v11,30,2026,2026,framework_filled_own_valid_approximation,return,12702,2726,113,70,0.088,0.138,67.74%,62,62,43.968,36.20%,-0.007,0.200,0.068,8.371,0.191,0.075,8,3,20,conditional_exploratory_block_interval


...

method_id,lookahead_days,period,cohort,outcome,rows,paired_rows,paired_vaults,origin_count,mean_ic,median_ic,positive_date_fraction,supported_origins,paired_origins,mean_paired_vaults,mean_tie_fraction,ic_lower,ic_upper,hac_se,effective_origins,minimum_detectable_ic,nonoverlap_mean_ic,nonoverlap_origins,independent_blocks,block_origins,inference_status
v11,7,2026,horizon_specific_origins,return,12702,3137,126,70,0.050,0.062,55.88%,68,68,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34,34,2,conditional_exploratory_block_interval
v11,7,2026,common_origins_all_horizons,return,8650,1895,92,47,0.074,0.115,59.57%,47,47,40.319,36.18%,0.019,0.133,0.031,34.154,0.088,0.105,24,23,2,conditional_exploratory_block_interval
v11,7,2026,common_origins_and_vaults_all_horizons,return,1796,1796,87,47,0.080,0.120,59.57%,47,47,38.213,36.16%,0.020,0.145,0.034,35.143,0.094,0.112,24,23,2,conditional_exploratory_block_interval
v11,30,2026,horizon_specific_origins,return,12702,2726,113,70,0.088,0.138,67.74%,62,62,43.968,36.20%,-0.007,0.200,0.068,8.371,0.191,0.075,8,3,20,conditional_exploratory_block_interval
v11,30,2026,common_origins_all_horizons,return,8650,1883,92,47,0.142,0.187,76.60%,47,47,40.064,36.37%,0.078,0.218,0.042,14.065,0.117,0.109,6,5,8,conditional_exploratory_block_interval
v11,30,2026,common_origins_and_vaults_all_horizons,return,1796,1796,87,47,0.147,0.182,76.60%,47,47,38.213,36.16%,0.081,0.229,0.044,13.788,0.123,0.125,6,5,8,conditional_exploratory_block_interval
v11,90,2026,horizon_specific_origins,return,12702,1796,87,70,0.184,0.239,76.60%,47,47,38.213,36.16%,—,—,—,—,—,0.054,3,2,23,short_90_day_descriptive_only
v11,90,2026,common_origins_all_horizons,return,8650,1796,87,47,0.184,0.239,76.60%,47,47,38.213,36.16%,—,—,—,—,—,0.054,3,2,23,short_90_day_descriptive_only
v11,90,2026,common_origins_and_vaults_all_horizons,return,1796,1796,87,47,0.184,0.239,76.60%,47,47,38.213,36.16%,—,—,—,—,—,0.054,3,2,23,short_90_day_descriptive_only
v11,7,2026,horizon_specific_origins,sharpe,12702,2848,123,70,0.063,0.084,69.12%,68,68,41.882,35.55%,0.013,0.107,0.025,43.344,0.071,0.086,34,34,2,conditional_exploratory_block_interval


### Market reference status — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Reference data availability, cache provenance and source fingerprints. Unavailable prices are reported explicitly without substituting another market series.

symbol,status,daily_candles,first_daily_label,last_daily_label,reference_sha256,timestamp_convention
BTCUSDT,available,3336,2017-08-17 00:00:00,2026-10-04 00:00:00,b2c351c468fc2fc2da820d308013ad86c3946477ff5285084596b8cfe99b6538,UTC open-labelled daily candle; its close is completed at label+1d.
ETHUSDT,available,3336,2017-08-17 00:00:00,2026-10-04 00:00:00,72420a1d850e25f9345e04d1aedb44f3df45ed4fd02aadac39f8ba7c77ac5710,UTC open-labelled daily candle; its close is completed at label+1d.


### Market adjusted summary — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Raw future log-return IC versus BTC/ETH exposure-adjusted IC on the identical beta-supported population. Betas use the past 90 daily returns with at least 60 supported calendar returns; actual paired observations are counted separately, with no future beta fitting or leading backfill.

method_id,lookahead_days,period,basis,outcome,rows,paired_rows,paired_vaults,origin_count,mean_ic,median_ic,positive_date_fraction,supported_origins,paired_origins,mean_paired_vaults,mean_tie_fraction,ic_lower,ic_upper,hac_se,effective_origins,minimum_detectable_ic,nonoverlap_mean_ic,nonoverlap_origins,independent_blocks,block_origins,inference_status
v11,7,2026,raw_all_eligible,log_return,12702,3137,126,70,0.050,0.062,55.88%,68,68,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34,34,2,conditional_exploratory_block_interval
v11,7,2026,raw_same_beta_supported_population,log_return,11136,3137,126,68,0.050,0.062,55.88%,68,68,46.132,36.59%,0.007,0.096,0.025,45.971,0.071,0.079,34,34,2,conditional_exploratory_block_interval
v11,7,2026,market_adjusted_same_population,log_return,11136,3137,126,68,0.085,0.089,75.00%,68,68,46.132,36.59%,0.042,0.138,0.028,34.028,0.080,0.098,34,17,4,conditional_exploratory_block_interval
v11,30,2026,raw_all_eligible,log_return,12702,2726,113,70,0.088,0.138,67.74%,62,62,43.968,36.20%,-0.007,0.200,0.068,8.371,0.191,0.075,8,3,20,conditional_exploratory_block_interval
v11,30,2026,raw_same_beta_supported_population,log_return,10102,2726,113,62,0.088,0.138,67.74%,62,62,43.968,36.20%,-0.007,0.200,0.068,8.371,0.191,0.075,8,3,20,conditional_exploratory_block_interval
v11,30,2026,market_adjusted_same_population,log_return,10102,2726,113,62,0.111,0.171,69.35%,62,62,43.968,36.20%,0.039,0.231,0.071,8.931,0.198,0.059,8,3,19,conditional_exploratory_block_interval
v11,90,2026,raw_all_eligible,log_return,12702,1796,87,70,0.184,0.239,76.60%,47,47,38.213,36.16%,—,—,—,—,—,0.054,3,2,23,short_90_day_descriptive_only
v11,90,2026,raw_same_beta_supported_population,log_return,7185,1796,87,47,0.184,0.239,76.60%,47,47,38.213,36.16%,—,—,—,—,—,0.054,3,2,23,short_90_day_descriptive_only
v11,90,2026,market_adjusted_same_population,log_return,7185,1796,87,47,0.217,0.273,85.11%,47,47,38.213,36.16%,—,—,—,—,—,0.192,3,2,23,short_90_day_descriptive_only
blend_46d,7,2026,raw_all_eligible,log_return,12702,3141,126,70,0.050,0.040,54.41%,68,68,46.191,28.32%,0.008,0.096,0.025,45.909,0.070,0.073,34,34,2,conditional_exploratory_block_interval


## Complete scatter catalogue

The complete **139-method × 7/30/90-day** catalogue is in [small linked notebooks](08-scatter-catalogue/index.ipynb). Every metric/view has a **separate image, heading and description**. There are no merged images or collations. Splitting the catalogue keeps notebook 08 usable in VS Code; its 27 main-method images remain visible above.

Each method/lookahead has a zoomed CAGR image, a full-range CAGR audit and a full-range Sharpe image. All charts use **2026-01-01 → 2026-10-05 (end exclusive, UTC)**. The table below lists every catalogue page and the methods it covers.

In [10]:
from score_forecast_catalogue import write_scatter_catalogue
with compact_output('Complete separate scatter catalogue — estimated 5–8 minutes'):
    scatter_manifest, catalogue_index = write_scatter_catalogue(data, registry_ids, FIGURE_DIR,
                                                              RESEARCH_DIR / '08-scatter-catalogue')
styled_table(catalogue_index, 'Complete scatter catalogue pages',
             'Open a linked page in 08-scatter-catalogue/index.ipynb to view the listed scoring recipes. '
             'Each small notebook embeds its own images and has executable cells to regenerate them.')


Complete separate scatter catalogue — estimated 5–8 minutes — complete

### Complete scatter catalogue pages — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Open a linked page in 08-scatter-catalogue/index.ipynb to view the listed scoring recipes. Each small notebook embeds its own images and has executable cells to regenerate them.

page,notebook,methods,images
1,08-scatter-01.ipynb,v11 (legacy 45d); Legacy 45d / 3 inputs / 1m CAGR; Legacy 45d / 3 inputs / 2m CAGR; Legacy 45d / 3 inputs / 3m CAGR,36
2,08-scatter-02.ipynb,Legacy 45d / 3 inputs / 6m CAGR; Legacy 45d / 3 inputs / 9m CAGR; Legacy 45d / 3 inputs / 2y CAGR; Legacy 45d / 3 inputs / 1m Sharpe,36
3,08-scatter-03.ipynb,Legacy 45d / 3 inputs / 3m Sharpe; Legacy 45d / 3 inputs / 6m Sharpe; Legacy 45d / 3 inputs / 1y Sharpe; Legacy 45d / 3 inputs / 1m Sortino,36
4,08-scatter-04.ipynb,Legacy 45d / 3 inputs / 3m Sortino; Legacy 45d / 3 inputs / 6m Sortino; Legacy 45d / 3 inputs / 1y Sortino; Legacy 45d / 3 inputs / 3m Max drawdown,36
5,08-scatter-05.ipynb,Legacy 45d / 3 inputs / 6m Max drawdown; Legacy 45d / 3 inputs / 3m Calmar; Legacy 45d / 3 inputs / 6m Calmar; Legacy 45d / 4 inputs / 1m CAGR + 3m Sharpe,36
6,08-scatter-06.ipynb,Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe; Legacy 45d / 4 inputs / 3m CAGR + 1y Sharpe; Four inputs (180d CAGR + 30d Sortino); Legacy 45d / 4 inputs / 9m CAGR + 3m Sortino,36
7,08-scatter-07.ipynb,Legacy 45d / 4 inputs / 2y CAGR + 6m Sortino; Legacy 45d / 4 inputs / 1m Sharpe + 1y Sortino; Legacy 45d / 4 inputs / 3m Sharpe + 3m Max drawdown; Legacy 45d / 4 inputs / 6m Sharpe + 6m Max drawdown,36
8,08-scatter-08.ipynb,Legacy 45d / 4 inputs / 1y Sharpe + 3m Calmar; Legacy 45d / 4 inputs / 1m Sortino + 6m Calmar; Legacy 45d / 4 inputs / 3m Sortino + 1m CAGR; Legacy 45d / 4 inputs / 6m Sortino + 2m CAGR,36
9,08-scatter-09.ipynb,Legacy 45d / 4 inputs / 1y Sortino + 3m CAGR; Legacy 45d / 4 inputs / 3m Max drawdown + 6m CAGR; Legacy 45d / 4 inputs / 6m Max drawdown + 9m CAGR; Legacy 45d / 4 inputs / 3m Calmar + 2y CAGR,36
10,08-scatter-10.ipynb,Legacy 45d / 4 inputs / 6m Calmar + 1m Sharpe; Blend 46d (two inputs); Blend 46d / 3 inputs / 1m CAGR; Blend 46d / 3 inputs / 2m CAGR,36


## Secondary history and completion record

Longer history is secondary. Its raw observations are sparse; primary metrics carry the latest known NAV and preserve actual observation counts separately.

In [11]:
styled_table(data['coverage'].query("period == 'Long'"), 'Full-history target availability',
             'All primary metrics use causally carried positive raw NAV. Actual observations remain identifiable; filled daily volatility uses a held-NAV assumption. '
             'Training uses only matured labels, with no leading backfill or target beyond each vault’s last actual observation.', period=LONG_PERIOD, collapsed=True)
styled_table(focal_ic_table(data, statistics, period='Long'), 'Secondary full-history predictive relationships',
             'The same recipes and origin grid extend backwards. More calendar history does not create more supported realised targets.', period=LONG_PERIOD, collapsed=True)
manifest = pd.DataFrame(scatter_manifest)
assert len(manifest) == 9 * len(registry_ids)
assert not manifest.duplicated(['method_id', 'lookahead_days', 'outcome', 'view']).any()
assert set(manifest.lookahead_days) == {7, 30, 90}
manifest.to_csv(ARTIFACT_DIR / 'scatter-figure-manifest.csv', index=False)
source_hashes = {p.name: __import__('hashlib').sha256(p.read_bytes()).hexdigest() for p in RESEARCH_DIR.glob('score_forecast_*.py')}
from score_forecast_catalogue import visual_review_status
review = visual_review_status(ARTIFACT_DIR)
completion = {'methods': len(registry_ids), 'scatter_figures': len(manifest), 'portfolio_arms': len(execution['results']), 'missing_score_cases': len(missing_sensitivity), 'additional_missing_score_replays': 6,
              'raw_label_audit': audit_report, 'source_hashes': source_hashes, 'data_fingerprint': data['provenance']['fingerprint'],
              'visual_review_complete': review['complete'], 'visual_review': review}
(ARTIFACT_DIR / 'completed.json').write_text(json.dumps(completion, indent=2))
findings = write_findings(data, statistics, execution, RESEARCH_DIR / '08-research-score-forecast-thresholds.ipynb')
styled_table(pd.DataFrame([{'Methods': len(registry_ids), 'Scatter figures': len(manifest),
                           'Portfolio arms': len(execution['results']), 'Raw label audit': audit_report['all_targets_match'],
                           'Visual inspection': 'Complete individual chart review' if review['complete'] else 'Pending complete chart review'}]),
             'Execution and evidence record', 'All figures and replay arms have saved manifests. The coordinator checks the complete rendered notebook '
             'and updates the heading with conclusions after execution; generated external artefacts remain ignored.')


### Full-history target availability — 2023-03-01 → 2026-10-05 (end exclusive, UTC)

All primary metrics use causally carried positive raw NAV. Actual observations remain identifiable; filled daily volatility uses a held-NAV assumption. Training uses only matured labels, with no leading backfill or target beyond each vault’s last actual observation.

period,lookahead_days,scheduled_origins,matured_origins,last_matured_origin,complete_pairs,return_endpoint_pairs,sharpe_finite_pairs,endpoint_only_pairs,framework_approximate_pairs,vaults_with_complete_target,complete_fraction_all_vault_origins,administrative_censoring_fraction,flat_target_fraction_complete,low_variance_fraction_complete
Long,7,329,327,2026-09-26 00:00:00,42189,42189,32095,0,42189,511,24.95%,0.61%,23.93%,26.73%
Long,30,329,321,2026-09-02 00:00:00,39160,39160,33393,0,39160,493,23.16%,2.43%,14.73%,17.92%
Long,90,329,306,2026-07-04 00:00:00,32062,32062,29358,0,32062,451,18.96%,6.99%,8.43%,11.73%


### Secondary full-history predictive relationships — 2023-03-01 → 2026-10-05 (end exclusive, UTC)

The same recipes and origin grid extend backwards. More calendar history does not create more supported realised targets.

method,lookahead_days,outcome,mean_ic,ic_lower,ic_upper,supported_origins,distinct_vaults,paired_rows,inference_status
Blend 46d (two inputs),7,cagr,0.070,0.027,0.113,122,130,4400,conditional_exploratory_block_interval
Blend 46d (two inputs),7,sharpe,0.080,0.041,0.114,110,127,3797,conditional_exploratory_block_interval
Blend 46d (two inputs),30,cagr,0.123,0.054,0.204,116,117,3987,conditional_exploratory_block_interval
Blend 46d (two inputs),30,sharpe,0.179,0.109,0.255,116,114,3842,conditional_exploratory_block_interval
Blend 46d (two inputs),90,cagr,0.245,0.217,0.308,101,92,3057,conditional_exploratory_block_interval
Blend 46d (two inputs),90,sharpe,0.252,0.165,0.359,101,91,3035,conditional_exploratory_block_interval
Four inputs (180d CAGR + 30d Sortino),7,cagr,0.097,0.055,0.142,113,130,4168,conditional_exploratory_block_interval
Four inputs (180d CAGR + 30d Sortino),7,sharpe,0.096,0.056,0.136,104,127,3671,conditional_exploratory_block_interval
Four inputs (180d CAGR + 30d Sortino),30,cagr,0.140,0.070,0.228,107,117,3770,conditional_exploratory_block_interval
Four inputs (180d CAGR + 30d Sortino),30,sharpe,0.186,0.112,0.269,106,114,3662,conditional_exploratory_block_interval


### Execution and evidence record — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

All figures and replay arms have saved manifests. The coordinator checks the complete rendered notebook and updates the heading with conclusions after execution; generated external artefacts remain ignored.

Methods,Scatter figures,Portfolio arms,Raw label audit,Visual inspection
139,1251,132,True,Pending complete chart review
